# 2. Data Cleaning and Target Construction

Build the panel from SEC annual filings, then engineer ratios and join annual FRED values.

**Verification:** code cells were statically checked. Raw SEC and FRED files are absent, so full extraction was not rerun. The supplied processed panel was checked separately for identifiers, adjacent-year features, labels and finite values. Revised extraction rules have not been verified against the omitted source files.

In [ ]:
from pathlib import Path
import sys

# Works from the project root or any folder beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/analysis_helpers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project folder.")
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

In [ ]:
"""Build the company-year panel from SEC quarterly bulk ZIPs.

Reads sub.txt (filings) and num.txt (XBRL facts) from each quarter, keeps only
annual 10-K filings for consolidated, parent-level, USD facts matching the
target tags, then pivots to one row per company-year.

Key detail: a 10-K also reports prior-year comparative figures, so a single tag
can appear with several `ddate` values. We only keep facts whose `ddate` equals
the filing's `period` (the current-year reported value) and exclude the
prior-year comparatives.
"""
import csv
import io
import os
import sys
import zipfile

import pandas as pd

RAW = ROOT / "data/raw"
PROC = ROOT / "data/processed"
os.makedirs(PROC, exist_ok=True)

# metric -> (ordered list of candidate tags, is_instant)
# The list order is the priority: earlier tags are preferred over later ones.
METRICS = {
    "revenue": (["Revenues", "SalesRevenueNet",
                 "RevenueFromContractWithCustomerExcludingAssessedTax",
                 "RevenueFromContractWithCustomerIncludingAssessedTax",
                 "SalesRevenueGoodsNet", "SalesRevenueServicesNet"], False),
    "operating_income": (["OperatingIncomeLoss"], False),
    "net_income": (["NetIncomeLoss", "ProfitLoss"], False),
    "total_assets": (["Assets"], True),
    "total_liabilities": (["Liabilities"], True),
    "current_assets": (["AssetsCurrent"], True),
    "current_liabilities": (["LiabilitiesCurrent"], True),
    "cash_and_equivalents": (["CashAndCashEquivalentsAtCarryingValue",
                              "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"], True),
    "operating_cash_flow": (["NetCashProvidedByUsedInOperatingActivities",
                             "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"], False),
    "capex": (["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets",
               "PaymentsForCapitalImprovements"], False),
}

TAG_TO_METRIC = {}
INSTANT = set()
for metric, (tags, is_inst) in METRICS.items():
    for t in tags:
        TAG_TO_METRIC[t] = metric
    if is_inst:
        INSTANT.add(metric)

# tag priority: position in the ordered list (lower is better)
TAG_PRIORITY = {}
for metric, (tags, _) in METRICS.items():
    for i, t in enumerate(tags):
        TAG_PRIORITY[t] = i


def read_quarter(qzip):
    """Read by column name: SEC layouts can add/reorder NUM fields."""
    with zipfile.ZipFile(qzip) as z:
        with z.open("sub.txt") as f:
            rows = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"), delimiter="\t")
            sub = pd.DataFrame([{k: r[k] for k in ("adsh", "cik", "name", "sic", "fy", "period", "filed")}
                                for r in rows if r.get("form") == "10-K" and r.get("fp") == "FY"])
        if sub.empty:
            return sub, pd.DataFrame()
        accessions = set(sub.adsh)
        facts = []
        with z.open("num.txt") as f:
            rows = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"), delimiter="\t")
            for row in rows:
                tag = row.get("tag")
                if row.get("adsh") not in accessions or tag not in TAG_TO_METRIC:
                    continue
                if row.get("segments", "").strip() or row.get("coreg", "").strip():
                    continue
                if row.get("uom") != "USD" or not row.get("version", "").startswith(("us-gaap", "ifrs")):
                    continue
                metric = TAG_TO_METRIC[tag]
                expected_qtrs = "0" if metric in INSTANT else "4"
                if row.get("qtrs") != expected_qtrs:
                    continue
                try:
                    value = float(row["value"])
                except (ValueError, TypeError):
                    continue
                if not __import__("math").isfinite(value):
                    continue
                facts.append({"adsh": row["adsh"], "tag": tag, "metric": metric,
                              "ddate": row["ddate"], "value": value,
                              "priority": TAG_PRIORITY[tag]})
    return sub, pd.DataFrame(facts)


def pivot_facts(facts, sub):
    """Current-period values only; conflicting equal-priority facts become missing."""
    if facts.empty:
        return pd.DataFrame(columns=["adsh", *METRICS])
    facts = facts.merge(sub[["adsh", "period"]], on="adsh", validate="many_to_one")
    facts = facts.loc[facts.ddate.eq(facts.period)].drop_duplicates()
    output = []
    for adsh, group in facts.groupby("adsh"):
        row = {"adsh": adsh}
        for metric in METRICS:
            candidates = group.loc[group.metric.eq(metric)]
            values = candidates.loc[candidates.priority.eq(candidates.priority.min()), "value"].unique()
            row[metric] = values[0] if len(values) == 1 else None
        output.append(row)
    return pd.DataFrame(output)


def main():
    qzips = sorted([os.path.join(RAW, f) for f in os.listdir(RAW) if f.endswith(".zip")])
    if not qzips:
        raise FileNotFoundError("Raw SEC ZIPs are required for Notebook 02; use Notebook 03 for the processed-data route.")

    subs, panels = [], []
    for i, qz in enumerate(qzips):
        q = os.path.basename(qz).replace(".zip", "")
        sub, facts = read_quarter(qz)
        panel = pivot_facts(facts, sub)
        subs.append(sub)
        panels.append(panel)
        print(f"{q}: {len(sub)} 10-K filings, {len(panel)} pivoted rows", flush=True)

    sub_all = pd.concat(subs, ignore_index=True)
    panel_all = pd.concat(panels, ignore_index=True)

    df = sub_all.merge(panel_all, on="adsh", how="inner")
    df["fiscal_year"] = pd.to_numeric(df["fy"], errors="coerce")
    df = df.dropna(subset=["fiscal_year"])
    df["fiscal_year"] = df["fiscal_year"].astype(int)
    df = df[df["fiscal_year"] >= 2012]
    df["cik"] = df["cik"].astype(str).str.zfill(10)

    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
    df = df.sort_values(["cik", "fiscal_year", "filed", "adsh"], ascending=[True, True, False, False], kind="mergesort")
    df = df.drop_duplicates(subset=["cik", "fiscal_year"], keep="first")

    cols = ["cik", "name", "sic", "fiscal_year"] + list(METRICS.keys())
    df = df[cols].sort_values(["cik", "fiscal_year"]).reset_index(drop=True)
    df = df.rename(columns={"name": "company_name"})

    out = os.path.join(PROC, "company_year_panel_raw.csv")
    df.to_csv(out, index=False)
    print(f"\nSaved {len(df)} rows to data/processed/company_year_panel_raw.csv")
    print("unique companies:", df["cik"].nunique())
    print("fiscal year range:", df["fiscal_year"].min(), "-", df["fiscal_year"].max())


main()

In [ ]:
"""Feature engineering + target construction.

Reads data/processed/company_year_panel_raw.csv, maps sectors, excludes
financials, builds engineered ratios, constructs the next-year pressure label,
and joins FRED macro variables. Writes data/processed/company_year_panel.csv.
"""
import os

import numpy as np
import pandas as pd

BASE = ROOT
PROC = os.path.join(BASE, "data", "processed")


def map_sector(sic):
    """Map SIC code to one of four sector groups, or None for excluded firms."""
    if pd.isna(sic):
        return None
    s = str(int(sic)).zfill(4)
    code = int(s[:2])
    if 60 <= code <= 67 or s == "6798":
        return None
    if 10 <= code <= 14 or code == 49:
        return "Energy / Capital-intensive"
    if code in (35, 36, 73):
        return "Technology / Software / Electronics"
    if 50 <= code <= 59:
        return "Consumer / Retail"
    if 15 <= code <= 17 or 20 <= code <= 39:
        return "Industrial / Manufacturing"
    return None


from analysis_helpers import build_target, rolling_margin_std


def main():
    df = pd.read_csv(os.path.join(PROC, "company_year_panel_raw.csv"), dtype={"cik": str})
    df["sector"] = df["sic"].apply(map_sector)
    df = df.dropna(subset=["sector"]).copy()

    # require a minimum revenue to avoid tiny shell firms whose ratios explode
    df = df[(df["revenue"] >= 1e8) & (df["total_assets"] >= 1e6)]

    # --- tag coverage audit ---
    raw_metrics = ["revenue", "operating_income", "net_income", "total_assets",
                   "total_liabilities", "current_assets", "current_liabilities",
                   "cash_and_equivalents", "operating_cash_flow", "capex"]
    print("=== Tag coverage audit (share of company-years with a non-null value) ===")
    print(df.groupby("sector")[raw_metrics].apply(lambda g: g.notna().mean().round(3)).to_string())

    # --- engineered features ---
    df = df.sort_values(["cik", "fiscal_year"]).reset_index(drop=True)

    df["log_assets"] = np.log1p(df["total_assets"])
    df["log_revenue"] = np.log1p(df["revenue"])

    # Level ratios first (do not depend on prior years)
    df["operating_margin"] = df["operating_income"] / df["revenue"]
    df["net_margin"] = df["net_income"] / df["revenue"]
    df["operating_cash_flow_to_assets"] = df["operating_cash_flow"] / df["total_assets"]
    df["cash_to_assets"] = df["cash_and_equivalents"] / df["total_assets"]
    df["liabilities_to_assets"] = df["total_liabilities"] / df["total_assets"]
    df["current_ratio"] = df["current_assets"] / df["current_liabilities"].where(df["current_liabilities"].gt(0))
    df["asset_turnover"] = df["revenue"] / df["total_assets"]
    df["capex_to_revenue"] = df["capex"] / df["revenue"]

    # Year-over-year changes must use the PREVIOUS fiscal year, not the previous
    # row. Prior-year values are kept only when the fiscal years are consecutive.
    g = df.groupby("cik")
    df["prev_year"] = g["fiscal_year"].shift(1)
    df["prev_revenue"] = g["revenue"].shift(1)
    df["prev_operating_margin"] = g["operating_margin"].shift(1)
    consecutive = df["fiscal_year"] == df["prev_year"] + 1

    df["revenue_growth"] = np.where(
        consecutive, df["revenue"] / df["prev_revenue"] - 1, np.nan)
    df["change_in_operating_margin"] = np.where(
        consecutive, df["operating_margin"] - df["prev_operating_margin"], np.nan)

    # Change in growth needs both the current and prior growth to be year-on-year.
    df["prev_revenue_growth"] = g["revenue_growth"].shift(1)
    df["change_in_revenue_growth"] = np.where(
        consecutive, df["revenue_growth"] - df["prev_revenue_growth"], np.nan)

    df["three_year_operating_margin_volatility"] = rolling_margin_std(df)

    # clean up temporary columns
    df = df.drop(columns=["prev_year", "prev_revenue", "prev_operating_margin",
                          "prev_revenue_growth"])

    # --- macro join ---
    df = df.merge(build_macro(), left_on="fiscal_year", right_on="year", how="left").drop(columns=["year"])

    # --- target ---
    df = build_target(df)

    # --- label prevalence check ---
    prev = df["next_year_financial_pressure"].mean() * 100
    print(f"\nLabel prevalence: {prev:.2f}%")
    if prev < 5 or prev > 40:
        print("WARNING: prevalence outside 5%-40%")
    else:
        print("Prevalence within the expected 5%-40% range.")

    # keep only companies with at least 3 rows (approximate 3 years of data)
    df = df.sort_values(["cik", "fiscal_year"])
    df["n_years"] = df.groupby("cik")["fiscal_year"].transform("count")
    df = df[df["n_years"] >= 3].drop(columns=["n_years"])

    out = os.path.join(PROC, "company_year_panel.csv")
    df["cik"] = df["cik"].astype(str).str.zfill(10)
    df = df.replace([np.inf, -np.inf], np.nan)
    df.to_csv(out, index=False)
    print(f"Saved {len(df)} rows to data/processed/company_year_panel.csv")
    print("companies:", df["cik"].nunique())
    print("sector counts:")
    print(df.groupby("sector")["cik"].nunique().to_string())
    print("pressure prevalence:", round(df["next_year_financial_pressure"].mean() * 100, 2), "%")
    print("fiscal year range:", df["fiscal_year"].min(), "-", df["fiscal_year"].max())


def build_macro():
    """Load FRED series and compute annual values."""
    def read_fred(name):
        p = os.path.join(BASE, "data", "raw", f"fred_{name}.csv")
        s = pd.read_csv(p)
        s.columns = ["date", "value"]
        s["date"] = pd.to_datetime(s["date"])
        s["value"] = pd.to_numeric(s["value"], errors="coerce")
        s = s.dropna()
        s["year"] = s["date"].dt.year
        counts = s.groupby("year")["date"].nunique()
        return s.loc[s["year"].isin(counts[counts.eq(12)].index)]

    fed = read_fred("FEDFUNDS").groupby("year")["value"].mean().rename("fed_funds_rate")
    cpi = read_fred("CPIAUCSL").groupby("year")["value"].mean()
    cpi_infl = cpi.pct_change(fill_method=None).where(cpi.index.to_series().diff().eq(1)) * 100
    cpi_infl = cpi_infl.rename("cpi_inflation")
    ind = read_fred("INDPRO").groupby("year")["value"].mean()
    ind_growth = ind.pct_change(fill_method=None).where(ind.index.to_series().diff().eq(1)) * 100
    ind_growth = ind_growth.rename("industrial_production_growth")

    return pd.concat([fed, cpi_infl, ind_growth], axis=1).reset_index()


main()

## Rules and limits

- Standard tags, USD, consolidated facts, current filing period, `qtrs=0` for instant values and `qtrs=4` for annual duration values. Conflicting best-priority facts are left missing. Fallback tags can differ in scope; see the [mapping](../data_dictionary/tag_mapping.csv).
- Keep annual 10-Ks, exclude amendments, and keep the latest 10-K per CIK/year (accession breaks filing-date ties). This is a retrospective panel, not a point-in-time filing backtest.
- Revenue at least $100M, assets at least $1M, four SIC groups, and at least three retained observations per company. Those observations need not be consecutive; lagged features and labels always require consecutive years.
- Margin volatility is population SD over the current and up to two preceding consecutive years, requiring at least two complete margins. One observation or a missing value in the window gives missing volatility.
- FRED values are contemporaneous calendar-year annual averages; CPI and production growth compare adjacent annual averages. Only complete 12-month years are used in a raw rebuild.
- Eligibility and source coverage can introduce selection bias. Processing more eligible firms does not remove it.

Outputs: the ignored raw intermediate and `company_year_panel.csv`.